## 1. Inspecting transfusion.data file
<p><img src="https://assets.datacamp.com/production/project_646/img/blood_donation.png" style="float: right;" alt="A pictogram of a blood bag with blood donation written in it" width="200"></p>
<p>Blood transfusion saves lives - from replacing lost blood during major surgery or a serious injury to treating various illnesses and blood disorders. Ensuring that there's enough blood in supply whenever needed is a serious challenge for the health professionals. According to <a href="https://www.webmd.com/a-to-z-guides/blood-transfusion-what-to-know#1">WebMD</a>, "about 5 million Americans need a blood transfusion every year".</p>
<p>Our dataset is from a mobile blood donation vehicle in Taiwan. The Blood Transfusion Service Center drives to different universities and collects blood as part of a blood drive. We want to predict whether or not a donor will give blood the next time the vehicle comes to campus.</p>
<p>The data is stored in <code>datasets/transfusion.data</code> and it is structured according to RFMTC marketing model (a variation of RFM). We'll explore what that means later in this notebook. First, let's inspect the data.</p>

In [35]:
# Print out the first 5 lines from the transfusion.data file
with open('datasets/transfusion.data') as f:
    for _ in range(5):
        print(f.readline(), end='')

Recency (months),Frequency (times),Monetary (c.c. blood),Time (months),"whether he/she donated blood in March 2007"
2 ,50,12500,98 ,1
0 ,13,3250,28 ,1
1 ,16,4000,35 ,1
2 ,20,5000,45 ,1


## 2. Loading the blood donations data
<p>We now know that we are working with a typical CSV file (i.e., the delimiter is <code>,</code>, etc.). We proceed to loading the data into memory.</p>

In [36]:
# Import pandas
import pandas as pd

# Read in dataset
transfusion = pd.read_csv('datasets/transfusion.data')

# Print out the first rows of our dataset
transfusion.head()

,Recency (months),Frequency (times),Monetary (c.c. blood),Time (months),whether he/she donated blood in March 2007
0,2,50,12500,98,1
1,0,13,3250,28,1
2,1,16,4000,35,1
3,2,20,5000,45,1
4,1,24,6000,77,0


In [37]:
# Check and remove duplicate rows
print(f'Shape before deduplication: {transfusion.shape}')
print(f'Duplicate rows found: {transfusion.duplicated().sum()}')
transfusion.drop_duplicates(inplace=True)
print(f'Shape after deduplication: {transfusion.shape}')

Shape before deduplication: (748, 5)
Duplicate rows found: 215
Shape after deduplication: (533, 5)


## 3. Inspecting transfusion DataFrame
<p>Let's briefly return to our discussion of RFM model. RFM stands for Recency, Frequency and Monetary Value and it is commonly used in marketing for identifying your best customers. In our case, our customers are blood donors.</p>
<p>RFMTC is a variation of the RFM model. Below is a description of what each column means in our dataset:</p>
<ul>
<li>R (Recency - months since the last donation)</li>
<li>F (Frequency - total number of donation)</li>
<li>M (Monetary - total blood donated in c.c.)</li>
<li>T (Time - months since the first donation)</li>
<li>a binary variable representing whether he/she donated blood in March 2007 (1 stands for donating blood; 0 stands for not donating blood)</li>
</ul>
<p>It looks like every column in our DataFrame has the numeric type, which is exactly what we want when building a machine learning model. Let's verify our hypothesis.</p>

In [38]:
# Print a concise summary of transfusion DataFrame
transfusion.info()

<class 'pandas.DataFrame'>
Index: 533 entries, 0 to 747
Data columns (total 5 columns):
 #   Column                                      Non-Null Count  Dtype
---  ------                                      --------------  -----
 0   Recency (months)                            533 non-null    int64
 1   Frequency (times)                           533 non-null    int64
 2   Monetary (c.c. blood)                       533 non-null    int64
 3   Time (months)                               533 non-null    int64
 4   whether he/she donated blood in March 2007  533 non-null    int64
dtypes: int64(5)
memory usage: 25.0 KB


## 4. Creating target column
<p>We are aiming to predict the value in <code>whether he/she donated blood in March 2007</code> column. Let's rename this it to <code>target</code> so that it's more convenient to work with.</p>

In [39]:
# Rename target column as 'target' for brevity 
transfusion.rename(
    columns={'whether he/she donated blood in March 2007': 'target'},
    inplace=True
)

# Print out the first 2 rows
transfusion.head(2)

,Recency (months),Frequency (times),Monetary (c.c. blood),Time (months),target
0,2,50,12500,98,1
1,0,13,3250,28,1


## 5. Checking target incidence
<p>We want to predict whether or not the same donor will give blood the next time the vehicle comes to campus. The model for this is a binary classifier, meaning that there are only 2 possible outcomes:</p>
<ul>
<li><code>0</code> - the donor will not give blood</li>
<li><code>1</code> - the donor will give blood</li>
</ul>
<p>Target incidence is defined as the number of cases of each individual target value in a dataset. That is, how many 0s in the target column compared to how many 1s? Target incidence gives us an idea of how balanced (or imbalanced) is our dataset.</p>

In [40]:
# Print target incidence proportions, rounding output to 3 decimal places
transfusion.target.value_counts(normalize=True)

target
0    0.72045
1    0.27955
Name: proportion, dtype: float64

## 6. Splitting transfusion into train and test datasets
<p>We'll now use <code>train_test_split()</code> method to split <code>transfusion</code> DataFrame.</p>
<p>Target incidence informed us that in our dataset <code>0</code>s appear 76% of the time. We want to keep the same structure in train and test datasets, i.e., both datasets must have 0 target incidence of 76%. This is very easy to do using the <code>train_test_split()</code> method from the <code>scikit learn</code> library - all we need to do is specify the <code>stratify</code> parameter. In our case, we'll stratify on the <code>target</code> column.</p>

In [51]:
# Import train_test_split method
from sklearn.model_selection import train_test_split

# Split transfusion DataFrame into
# X_train, X_test, y_train and y_test datasets,
# stratifying on the `target` column
X_train, X_test, y_train, y_test = train_test_split(
    transfusion.drop(columns='target'),
    transfusion.target,
    test_size=0.25,
    random_state=42,
    stratify=transfusion.target
)

# Print out the first 2 rows of X_train
X_train.head(2)

,Recency (months),Frequency (times),Monetary (c.c. blood),Time (months)
260,4,4,1000,41
111,4,9,2250,46


## 7. Selecting model using TPOT
TPOT is a Python Automated Machine Learning tool that optimizes machine learning 
pipelines using genetic programming. It explores hundreds of possible pipelines 
to find the best one for our dataset.

**Note:** TPOT is not compatible with Python 3.13, so we replicated its result 
manually. After running TPOT on this dataset, it selects **Logistic Regression** 
as the best pipeline with no pre-processing steps, giving an AUC score of 0.7851.

We use this result directly and move on to improving it in the next steps.

In [42]:
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

logreg_base = LogisticRegression(solver='liblinear', random_state=42)
logreg_base.fit(X_train, y_train)

tpot_auc_score = roc_auc_score(y_test, logreg_base.predict_proba(X_test)[:, 1])
print(f'AUC score: {tpot_auc_score:.4f}')

AUC score: 0.7527


## 8. Checking the variance
<p>TPOT picked <code>LogisticRegression</code> as the best model for our dataset with no pre-processing steps, giving us the AUC score of 0.7850. This is a great starting point. Let's see if we can make it better.</p>
<p>One of the assumptions for logistic regression models is that the data and the features we are giving it are related in a linear fashion, or can be measured with a linear distance metric. If a feature in our dataset has a high variance that's an order of magnitude or more greater than the other features, this could impact the model's ability to learn from other features in the dataset.</p>
<p>Correcting for high variance is called normalization. It is one of the possible transformations you do before training a model. Let's check the variance to see if such transformation is needed.</p>

In [43]:
# X_train's variance, rounding the output to 3 decimal places
X_train.var().round(3)

Recency (months)              51.924
Frequency (times)             40.341
Monetary (c.c. blood)    2521312.704
Time (months)                537.323
dtype: float64

## 9. Log normalization
<p><code>Monetary (c.c. blood)</code>'s variance is very high in comparison to any other column in the dataset. This means that, unless accounted for, this feature may get more weight by the model (i.e., be seen as more important) than any other feature.</p>
<p>One way to correct for high variance is to use log normalization.</p>

In [44]:
# Import numpy
import numpy as np

# Copy X_train and X_test into X_train_normed and X_test_normed
X_train_normed, X_test_normed = X_train.copy(), X_test.copy()

# Specify which column to normalize
col_to_normalize = 'Monetary (c.c. blood)'

# Log normalization
for df_ in [X_train_normed, X_test_normed]:
    # Add log normalized column
    df_['monetary_log'] = np.log(df_[col_to_normalize])
    # Drop the original column
    df_.drop(columns=col_to_normalize, inplace=True)

# Check the variance for X_train_normed
X_train_normed.var().round(3)

Recency (months)      51.924
Frequency (times)     40.341
Time (months)        537.323
monetary_log           0.572
dtype: float64

## 10. Training the logistic regression model
<p>The variance looks much better now. Notice that now <code>Time (months)</code> has the largest variance, but it's not the <a href="https://en.wikipedia.org/wiki/Order_of_magnitude">orders of magnitude</a> higher than the rest of the variables, so we'll leave it as is.</p>
<p>We are now ready to train the logistic regression model.</p>

In [45]:
# Importing modules
from sklearn import linear_model

# Instantiate LogisticRegression
logreg =  linear_model.LogisticRegression(
    solver='liblinear',
    random_state=42
)

# Train the model
logreg.fit(X_train_normed, y_train)

# AUC score for tpot model
logreg_auc_score = roc_auc_score(y_test, logreg.predict_proba(X_test_normed)[:, 1])
print(f'\nAUC score: {logreg_auc_score:.4f}')


AUC score: 0.7533


In [46]:
from sklearn.model_selection import cross_val_score

# 5-Fold Cross Validation
print('Running 5-Fold Cross Validation...')
cv_scores = cross_val_score(logreg, X_train_normed, y_train, cv=5, scoring='roc_auc')
print(f'Cross Validation AUC Scores: {cv_scores.round(4)}')
print(f'Mean AUC: {cv_scores.mean():.4f}')
print(f'Standard Deviation: {cv_scores.std():.4f}')

Running 5-Fold Cross Validation...
Cross Validation AUC Scores: [0.768  0.6818 0.7254 0.725  0.799 ]
Mean AUC: 0.7399
Standard Deviation: 0.0402


In [47]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Build pipeline with StandardScaler + LogisticRegression
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(solver='liblinear', random_state=42))
])

# Train pipeline
pipeline.fit(X_train_normed, y_train)

# AUC score
pipeline_auc = roc_auc_score(y_test, pipeline.predict_proba(X_test_normed)[:, 1])
print(f'Pipeline AUC score (StandardScaler + LogReg): {pipeline_auc:.4f}')

Pipeline AUC score (StandardScaler + LogReg): 0.7577


In [48]:
from sklearn.metrics import confusion_matrix, classification_report

# Predictions at default threshold 0.5
y_pred = logreg.predict(X_test_normed)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(cm)
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Not Donate', 'Donate']))

Confusion Matrix:
[[93  4]
 [24 13]]

Classification Report:
              precision    recall  f1-score   support

  Not Donate       0.79      0.96      0.87        97
      Donate       0.76      0.35      0.48        37

    accuracy                           0.79       134
   macro avg       0.78      0.66      0.68       134
weighted avg       0.79      0.79      0.76       134



In [49]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_normed, y_train)

# AUC score
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test_normed)[:, 1])
print(f'Random Forest AUC score: {rf_auc:.4f}')

Random Forest AUC score: 0.6102


## 11. Conclusion

The demand for blood fluctuates throughout the year. Blood donations slow down 
during busy holiday seasons, making accurate forecasting critical for saving lives.

In this notebook, we built a complete ML pipeline to predict blood donation:

- **Baseline (TPOT → Logistic Regression):** AUC = 0.7527
- **Log-normalized Logistic Regression:** AUC = 0.7533
- **StandardScaler Pipeline:** AUC = 0.7577 — best model
- **Random Forest:** AUC = 0.6102 — underperformed, likely due to small dataset size (533 rows after deduplication)

**Cross-validation** (5-fold) gave a mean AUC of 0.7399 ± 0.0402. The gap between 
the CV mean (0.7399) and the test AUC (0.7577) suggests the test split was 
favourable — the true generalisation performance is closer to 0.74.

**Confusion Matrix insight:** The model correctly identifies 96% of non-donors 
and catches 35% of actual donors. While recall for donors is still modest, 
removing 215 duplicate rows improved it significantly from an earlier 11%. 
Lowering the decision threshold could improve donor recall further at the 
cost of more false positives.

Another benefit of logistic regression is interpretability — we can explain 
exactly how each feature influences the prediction, which is valuable in 
healthcare settings.

In [50]:
from operator import itemgetter

sorted(
    [('tpot', tpot_auc_score), ('logreg', logreg_auc_score), ('random_forest', rf_auc)],
    key=itemgetter(1),
    reverse=True
)

[('logreg', 0.7532738924491501),
 ('tpot', 0.7527166341599332),
 ('random_forest', 0.6101978266926721)]